# Neural Deconstruction Search for CVRP: Paper Reproduction

**Paper:** *Neural Deconstruction Search for Vehicle Routing Problems* (Hottung et al., 2025)

**Scope:**

## Setup

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from core.problems.cvrp import CVRPBatchSampler, CVRPConfig
from core.problems.cvrp.solve import one_customer_per_route

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"PyTorch version: {torch.__version__}")
print(f"Using device: {DEVICE}")

PyTorch version: 2.10.0+cu130
Using device: cuda


In [3]:
%load_ext autoreload
%autoreload 2

## Data

In [4]:
example_sampler = CVRPBatchSampler(CVRPConfig.for_n(20))

example_batch = example_sampler(4)
print(example_batch)

CVRPBatchInstance(
    capacity=Tensor(shape=torch.Size([4, 1]), device=cpu, dtype=torch.float32, is_shared=False),
    demand=Tensor(shape=torch.Size([4, 21]), device=cpu, dtype=torch.float32, is_shared=False),
    locations=Tensor(shape=torch.Size([4, 21, 2]), device=cpu, dtype=torch.float32, is_shared=False),
    batch_size=torch.Size([4]),
    device=None,
    is_shared=False)


## Model Architecture

In [5]:
D_H = 128
N_HEADS = 8
D_FF = 512

## Encoder

**Initial Node Embedding**

In [6]:
class CVRPInitialNodeEmbedding(nn.Module):
    """
    Separate linear projections for depot and customer nodes.

    Depot: (x, y) -> d_h
    Customer: (x, y, demand) -> d_h
    """

    def __init__(self, d_h: int = D_H) -> None:
        super().__init__()
        self.depot_projection = nn.Linear(2, d_h)  # (x, y)
        self.customer_projection = nn.Linear(3, d_h)  # (x, y, demand)

    def forward(
        self,
        depot_location: torch.Tensor,  # (B, 1, 2)
        customer_features: torch.Tensor,  # (B, N, 3)
    ) -> torch.Tensor:  # (B, N+1, d_h)
        depot_embedding = self.depot_projection(depot_location)
        customer_embeddings = self.customer_projection(customer_features)
        return torch.cat([depot_embedding, customer_embeddings], dim=1)


depot_location = example_batch.depot_location
customer_features = example_batch.customer_features

print(f"Depot location shape: {depot_location.shape}")
print(f"Customer features shape: {customer_features.shape}")

initial_embedding = CVRPInitialNodeEmbedding()
initial_node_embeddings = initial_embedding(depot_location, customer_features)
print(f"Initial node embeddings shape: {initial_node_embeddings.shape}")

Depot location shape: torch.Size([4, 1, 2])
Customer features shape: torch.Size([4, 20, 3])
Initial node embeddings shape: torch.Size([4, 21, 128])


**Attention Layer**

Same as in "Attention, Learn to Solve Routing Problems!" (Kool et al., 2019).

In [7]:
class AttentionLayer(nn.Module):
    def __init__(
        self,
        d_h: int = D_H,
        n_heads: int = N_HEADS,
        d_ff: int = D_FF,
    ) -> None:
        super().__init__()
        self.MHA = nn.MultiheadAttention(d_h, n_heads, batch_first=True)
        self.norm1 = nn.BatchNorm1d(d_h)
        self.FF = nn.Sequential(
            nn.Linear(d_h, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_h),
        )
        self.norm2 = nn.BatchNorm1d(d_h)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, L, d = x.shape
        attn_out, _ = self.MHA(x, x, x)
        x = self.norm1((x + attn_out).reshape(B * L, d)).reshape(B, L, d)
        x = self.norm2((x + self.FF(x)).reshape(B * L, d)).reshape(B, L, d)
        return x


att_layer = AttentionLayer()
refined_embeddings = att_layer(initial_node_embeddings)

print(f"Refined node embeddings shape: {refined_embeddings.shape}")

Refined node embeddings shape: torch.Size([4, 21, 128])


**Message Passing Layer**

In [8]:
class MessagePassingLayer(nn.Module):
    def __init__(self, d_h: int = D_H, d_ff: int = D_FF) -> None:
        super().__init__()
        self.W1 = nn.Linear(d_h, d_h, bias=False)  # previous neighbour projection
        self.W2 = nn.Linear(d_h, d_h, bias=False)  # next neighbour projection
        self.W3 = nn.Linear(2 * d_h, d_h, bias=False)
        self.FF = nn.Sequential(
            nn.Linear(d_h, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_h),
        )
        self.norm = nn.LayerNorm(d_h)

    def forward(
        self,
        h: torch.Tensor,
        neighbours_index: torch.Tensor,
    ) -> torch.Tensor:
        """
        Args:
            h: (B, N+1, d_h) node embeddings, where h[:, 0] is the depot embedding and h[:, 1:] are customer embeddings
            neighbours_index: (B, N, 2) integer indices of the previous and next nodes for each customer

        Returns:
            (B, N+1, d_h) updated node embeddings after message passing
        """
        customer_embeddings = h[:, 1:]  # (B, N, d_h)

        previous = self._gather(h, neighbours_index[:, :, 0])  # (B, N, d_h)
        next = self._gather(h, neighbours_index[:, :, 1])  # (B, N, d_h)

        prev_emb = self.W1(previous)
        next_emb = self.W2(next)
        agg = prev_emb + next_emb

        update = self.FF(F.relu(self.W3(torch.cat([customer_embeddings, agg], dim=-1))))
        updated = self.norm(customer_embeddings + update)

        return torch.cat([h[:, :1], updated], dim=1)

    def _gather(self, embeddings: torch.Tensor, indices: torch.Tensor) -> torch.Tensor:
        """
        Args:
            embeddings: (B, N+1, d_h)
            indices: (B, N)

        Returns:
            (B, N, d_h)
        """
        B, N = indices.shape
        d_h = embeddings.shape[2]
        return embeddings.gather(1, indices[:, :, None].expand(B, N, d_h))

**Tour Encoding Layer**

In [9]:
# def native_scatter_mean(
#     src: torch.Tensor,
#     index: torch.Tensor,
#     dim: int = 0,
#     dim_size: int | None = None
# ) -> torch.Tensor:
#     if dim_size is None:
#         dim_size = int(index.max().item()) + 1
#
#     out_shape = list(src.shape)
#     out_shape[dim] = dim_size
#     out = torch.zeros(out_shape, device=src.device, dtype=src.dtype)
#
#     if index.ndim < src.ndim:
#         view_shape = [1] * src.ndim
#         view_shape[dim] = index.shape[dim]
#         expanded_index = index.view(view_shape).expand_as(src)
#     else:
#         expanded_index = index
#
#     return out.scatter_reduce(
#         dim=dim,
#         index=expanded_index,
#         src=src,
#         reduce="mean",
#         include_self=False
#     )

In [10]:
class TourEncodingLayer(nn.Module):
    def __init__(self, d_h: int = D_H, d_ff: int = D_FF) -> None:
        super().__init__()
        self.tour_combiner = nn.Linear(2 * d_h, d_h, bias=False)
        self.FF = nn.Sequential(
            nn.Linear(d_h, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_h),
        )
        self.norm = nn.LayerNorm(d_h)

    def forward(
        self,
        node_embeddings: torch.Tensor,
        tour_index: torch.Tensor,
    ) -> torch.Tensor:
        """
        Args:
            node_embeddings: (B, N+1, d_h)
            tour_index: (B, N)

        Returns:
            (B, N+1, d_h)
        """
        B, L, d_h = node_embeddings.shape
        N = L - 1

        tour_index = tour_index.reshape(B, N)

        customer_embeddings = node_embeddings[:, 1:]

        max_tours = int(tour_index.max().item()) + 1
        idx = tour_index.unsqueeze(-1).expand(B, N, d_h)
        tour_embeddings = torch.zeros(
            B, max_tours, d_h, device=node_embeddings.device, dtype=node_embeddings.dtype
        )
        tour_embeddings = tour_embeddings.scatter_reduce(
            1, idx, customer_embeddings, reduce="mean", include_self=False
        )
        tour_mean_per_customer = tour_embeddings.gather(1, idx)  # (B, N, d_h)

        update = self.FF(
            F.relu(
                self.tour_combiner(torch.cat([customer_embeddings, tour_mean_per_customer], dim=-1))
            )
        )
        updated_customer_embeddings = self.norm(customer_embeddings + update)

        return torch.cat(
            [node_embeddings[:, :1], updated_customer_embeddings], dim=1
        )  # (B, N+1, d_h)

**All sub-layers combined into the encoder**

In [11]:
class NDSEncoder(nn.Module):
    def __init__(
        self,
        d_h: int = D_H,
        n_heads: int = N_HEADS,
        d_ff: int = D_FF,
    ) -> None:
        super().__init__()

        self.embedding = CVRPInitialNodeEmbedding(d_h)
        attention_layer = lambda: AttentionLayer(d_h, n_heads, d_ff)
        self.pre_attention = nn.Sequential(
            attention_layer(),
            attention_layer(),
        )
        self.message_passing = MessagePassingLayer(d_h, d_ff)
        self.tour_encoding = TourEncodingLayer(d_h, d_ff)
        self.post_attention = nn.Sequential(
            attention_layer(),
            attention_layer(),
        )

    def forward(
        self,
        *,
        depot_location: torch.Tensor,
        customer_features: torch.Tensor,
        neighbours_index: torch.Tensor,
        tour_index: torch.Tensor,
    ) -> torch.Tensor:
        """
        Args:
            depot_location: (B, 2)
            customer_features: (B, N, 3)
            neighbours_index: (B, N, 2)
            tour_index: (B, N)

        Returns:
            (B, N+1, d_h)
        """
        h = self.embedding(depot_location, customer_features)
        h = self.pre_attention(h)
        h = self.message_passing(h, neighbours_index)
        h = self.tour_encoding(h, tour_index)
        h = self.post_attention(h)
        return h

In [12]:
def build_encoder_input_indexes_from_routes(
    routes_batch: list[list[torch.Tensor]],
    num_customers: int,
    device: torch.device = DEVICE,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Returns:
        neighbours_index: (B, N, 2) [prev, next] node index per customer, 1-indexed
        tour_index: (B, N) local tour id per customer, 0-indexed
    """
    B = len(routes_batch)
    N = num_customers

    neighbours = torch.zeros(B, N, 2, dtype=torch.long, device=device)
    tour_index = torch.zeros(B, N, dtype=torch.long, device=device)

    for b_idx, routes in enumerate(routes_batch):
        tour_with_depot = [[0, *route, 0] for route in routes]
        for tour_idx, tour in enumerate(tour_with_depot):
            for pos in range(1, len(tour) - 1):
                customer = tour[pos]
                tour_index[b_idx, customer - 1] = tour_idx
                neighbours[b_idx, customer - 1] = torch.tensor([tour[pos - 1], tour[pos + 1]])

    return neighbours, tour_index

In [13]:
def build_encoder_input_indexes_from_sequences(
    sequences: torch.Tensor,  # (B, L)
    num_customers: int,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Parse flat sequences into encoder inputs.

    Returns:
        neighbours: (B, N, 2)
        route_index: (B, N)
    """
    B = sequences.shape[0]
    N = num_customers
    device = sequences.device

    neighbours = torch.zeros(B, N, 2, dtype=torch.long, device=device)
    tour_index = torch.zeros(B, N, dtype=torch.long, device=device)

    for b in range(B):
        seq = sequences[b]
        depot_positions = (seq == 0).nonzero(as_tuple=True)[0]
        for tour_idx in range(len(depot_positions) - 1):
            start = depot_positions[tour_idx].item()
            end = depot_positions[tour_idx + 1].item()
            tour = seq[start : end + 1]  # includes depot on both sides
            for pos in range(1, len(tour) - 1):
                customer = tour[pos].item()
                tour_index[b, customer - 1] = tour_idx
                neighbours[b, customer - 1] = torch.tensor(
                    [tour[pos - 1].item(), tour[pos + 1].item()]
                )

    return neighbours, tour_index

In [14]:
encoder = NDSEncoder().to(DEVICE)

# Batch: B=2, N=5 customers
example_sampler = CVRPBatchSampler(
    CVRPConfig(num_customers=5, vehicle_capacity_raw=9),
    seed=42,
)
batch = example_sampler(2).to(DEVICE)
depot = batch.depot_location
customers = batch.customer_features
# routes = [[[1,3,2], [4,5]], [[2,1,4], [3,5]]]
# neighbours_index, tour_index = build_encoder_input_indexes_from_routes(routes, num_customers=batch.num_customers)

batch_solution = one_customer_per_route(batch)
print(batch_solution.sequences)

neighbours_index, tour_index = build_encoder_input_indexes_from_sequences(
    batch_solution.sequences, num_customers=batch.num_customers
)

h = encoder(
    depot_location=depot,
    customer_features=customers,
    neighbours_index=neighbours_index,
    tour_index=tour_index,
)

print(f"Encoder output shape: {h.shape}")

tensor([[0, 1, 0, 2, 0, 3, 0, 4, 0, 5, 0],
        [0, 1, 0, 2, 0, 3, 0, 4, 0, 5, 0]], device='cuda:0')
Encoder output shape: torch.Size([2, 6, 128])


## Training